In [10]:
%gui qt
import napari
import numpy as np
import tifffile
import scipy as sp
import cv2
#import cupyx.scipy.ndimage as ndi
import scipy.ndimage as ndi
from skimage.transform import rescale, resize, downscale_local_mean
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go

In [4]:
datas = ['S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/',
               'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210909_MER_20-15-5B_4dpi/cjw/DeepLearn/',
               'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210129_MER_20-15-7b_8dpi_stack2/cjw/DeepLearn/',
               'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210326_MER_20-15-8a_14dpi/cjw/DeepLearn/',
        'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20220107_MER_IMARE-110171_19_55_48hpa_C4/cjw/gcs_etomo/DeepLearn/']

data_fix = ['control', '4dpi', '8dpi', '14dpi', '48hpa']

In [16]:
def get_data(current_path, dataset):
    labels = tifffile.imread(current_path+'Labels.tif')
    max_count = np.max(labels)
    volumes = ndi.labeled_comprehension(labels, labels, np.arange(1,max_count+1), len, float, 0)
    centers_of_mass = ndi.center_of_mass(labels[::4,::4,::4], labels[::4,::4,::4], np.arange(1,max_count+1))
    centers_of_mass = np.array(centers_of_mass)*4
    
    rtn = pd.DataFrame({'Dataset':dataset, 'Path':current_path, 'ID':(np.arange(1,max_count+1)-1), 'Volume':volumes, 
                        'Z':centers_of_mass[:,0],'Y':centers_of_mass[:,1], 'X':centers_of_mass[:,2]})
    return rtn

In [19]:
df = pd.DataFrame()
for a in range(0,len(datas)):
    tdf = get_data(datas[a], data_fix[a])
    df = pd.concat([df,tdf])

In [21]:
df.to_csv('Volume-CMs.csv')